In [0]:
%run ./01-config

In [0]:
# Import PySpark SQL functions for transformations (e.g., current_timestamp, input_file_name, to_date)
from pyspark.sql import functions as F
from pyspark.sql.functions import col, current_timestamp

In [0]:
#   Construct absolute paths for raw data ingestion and streaming checkpoints
# - `landing_zone`: Base directory where raw source files (CSV/JSON) are staged for ingestion

base_dir_data = spark.sql("describe external location `data-zone`").select("url").collect()[0][0]
base_dir_checkpoint = spark.sql("describe external location `checkpoints-zone`").select("url").collect()[0][0]

In [0]:
# # Create catalog
spark.sql("""
CREATE CATALOG IF NOT EXISTS dev
""")

In [0]:
# # Create bronze schema
spark.sql(f"""
    CREATE DATABASE IF NOT EXISTS bronze
    MANAGED LOCATION '{base_dir_data}/bronze'
""")


In [0]:
catalog = "dev"
schema_1 = "bronze"


In [0]:
# - `checkpoint_base`: Directory to store streaming query offsets and state (required for fault tolerance)

TRAIN_CHECKPOINT = base_dir_checkpoint + "/bronze/train_movements_raw"
WEATHER_CHECKPOINT = base_dir_checkpoint + "/bronze/weather_raw"
CITY_ROUTE_CHECKPOINT = base_dir_checkpoint + "/bronze/city_routes_raw"
STATION_CHECKPOINT = base_dir_checkpoint + "/bronze/station_raw"
STATIONS_CODE_CHECKPOINT = base_dir_checkpoint + "/bronze/stations_code_raw"

## Consumer

In [0]:

# --------------------------------------------------------------------------------------------------
# BRONZE LAYER STREAMING INGESTION FUNCTIONS
# Each function consumes raw files from cloud storage and writes them as Delta tables in append-only mode.
# --------------------------------------------------------------------------------------------------

def stream_to_bronze_train(
    topic: str,
    API_KEY: str,
    API_SECRET: str,
    catalog: str,
    schema_1: str,
    checkpoint_dir: str,
    trigger_once: bool = True,
    processing_time: str = "10 seconds"
):
    """
    Streams raw Kafka data into a Bronze Delta table.

    Bronze principles:
    - No schema enforcement
    - Raw JSON payload preserved
    - Kafka metadata retained
    """

    # -----------------------
    # Kafka Configuration
    # -----------------------
    kafka_df = (
        spark.readStream
            .format("kafka")
            .option("kafka.bootstrap.servers", "pkc-12p03.southafricanorth.azure.confluent.cloud:9092")
            .option("subscribe", topic)
            .option("startingOffsets", "earliest")
            .option("kafka.security.protocol", "SASL_SSL")
            .option("kafka.sasl.mechanism", "PLAIN")
            .option(
                "kafka.sasl.jaas.config",
                f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
                f'username="{API_KEY}" password="{API_SECRET}";'
            )
            .load()
    )

    # -----------------------
    # Bronze Transformation
    # -----------------------
    bronze_df = (
        kafka_df
            .select(
                col("topic"),
                col("partition"),
                col("offset"),
                col("timestamp").alias("kafka_timestamp"),
                col("key").cast("string").alias("kafka_key"),
                col("value").cast("string").alias("raw_payload")
            )
            .withColumn("ingestion_time", current_timestamp())
    )

    # -----------------------
    # Write to Bronze Delta
    # -----------------------
    writer = (
        bronze_df.writeStream
            .format("delta")
            .outputMode("append")
            .option("checkpointLocation", checkpoint_dir)
            .queryName(f"bronze_{topic}")
    )

    if trigger_once:
        return writer.trigger(availableNow=True).toTable(f"{catalog}.{schema_1}.train_movements_raw")
    else:
        return writer.trigger(processingTime=processing_time).toTable(f"{catalog}.{schema_1}.train_movements_raw")


In [0]:
# --------------------------------------------------------------------------------------------------
# BRONZE LAYER STREAMING INGESTION FUNCTIONS
# Each function consumes raw files from cloud storage and writes them as Delta tables in append-only mode.
# --------------------------------------------------------------------------------------------------

def stream_to_bronze_weather(
    topic: str,
    API_KEY: str,
    API_SECRET: str,
    catalog: str,
    schema_1: str,
    checkpoint_dir: str,
    trigger_once: bool = True,
    processing_time: str = "10 seconds"
):
    """
    Streams raw Kafka data into a Bronze Delta table.

    Bronze principles:
    - No schema enforcement
    - Raw JSON payload preserved
    - Kafka metadata retained
    """

    # -----------------------
    # Kafka Configuration
    # -----------------------
    kafka_df = (
        spark.readStream
            .format("kafka")
            .option("kafka.bootstrap.servers", "pkc-12p03.southafricanorth.azure.confluent.cloud:9092")
            .option("subscribe", topic)
            .option("startingOffsets", "earliest")
            .option("kafka.security.protocol", "SASL_SSL")
            .option("kafka.sasl.mechanism", "PLAIN")
            .option(
                "kafka.sasl.jaas.config",
                f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
                f'username="{API_KEY}" password="{API_SECRET}";'
            )
            .load()
    )

    # -----------------------
    # Bronze Transformation
    # -----------------------
    bronze_df = (
        kafka_df
            .select(
                col("topic"),
                col("partition"),
                col("offset"),
                col("timestamp").alias("kafka_timestamp"),
                col("key").cast("string").alias("kafka_key"),
                col("value").cast("string").alias("raw_payload")
            )
            .withColumn("ingestion_time", current_timestamp())
    )

    # -----------------------
    # Write to Bronze Delta
    # -----------------------
    writer = (
        bronze_df.writeStream
            .format("delta")
            .outputMode("append")
            .option("checkpointLocation", checkpoint_dir)
            .queryName(f"bronze_{topic}")
    )

    if trigger_once:
        return writer.trigger(availableNow=True).toTable(f"{catalog}.{schema_1}.weather_raw")
    else:
        return writer.trigger(processingTime=processing_time).toTable(f"{catalog}.{schema_1}.weather_raw")

In [0]:


# --------------------------------------------------------------------------------------------------
# BRONZE LAYER STREAMING INGESTION FUNCTIONS
# Each function consumes raw files from cloud storage and writes them as Delta tables in append-only mode.
# --------------------------------------------------------------------------------------------------

def consume_city_routes(catalog: str, schema_1: str, checkpoint_dir: str, once=True, processing_time="5 seconds"):
    """
    Ingests city-route reference CSV files into a Bronze Delta table `city_routes_bronze`.
    Adds metadata columns (`load_time`, `source_file`) for auditability.

    Parameters:
      once (bool): If True, runs as a batch (trigger once); if False, runs as a continuous stream.
      processing_time (str): Interval for micro-batch triggers when `once=False`.
    """

    # -------------------------------
    # Define expected CSV schema
    # -------------------------------
    schema = "City STRING, Route STRING"

    # -------------------------------
    # Auto Loader stream
    # -------------------------------
    df_stream = (
        spark.readStream
            .format("cloudfiles")                     # Databricks Auto Loader
            .schema(schema)                           # Explicit schema
            .option("cloudFiles.maxFilesPerTrigger", 1)  # 1 file per micro-batch
            .option("cloudFiles.format", "csv")       # Source format
            .option("header", True)                   # First row contains column names
            .load(base_dir_data + "/city_route")  # Source path
            .withColumn("load_time", F.current_timestamp())   # Track ingestion time
            .withColumn("source_file", F.input_file_name())  # Track source file
    )

    # -------------------------------
    # Write stream to Bronze Delta table
    # -------------------------------
    

    stream_writer = (
        df_stream.writeStream
            .format("delta")
            .option("checkpointLocation", checkpoint_dir)
            .outputMode("append")
            .queryName("bronze_city_routes")  # Unique name for monitoring
    )

    # -------------------------------
    # Trigger execution
    # -------------------------------
    if once:
        return stream_writer.trigger(availableNow=True).toTable(f"{catalog}.{schema_1}.city_routes_raw")
    else:
        return stream_writer.trigger(processingTime=processing_time).toTable(f"{catalog}.{schema_1}.city_routes_raw")


In [0]:


# --------------------------------------------------------------------------------------------------
# BRONZE LAYER STREAMING INGESTION FUNCTIONS
# Each function consumes raw files from cloud storage and writes them as Delta tables in append-only mode.
# --------------------------------------------------------------------------------------------------


def consume_station(catalog: str, schema_1: str, checkpoint_dir: str, once=True, processing_time="5 seconds"):
    """
    Ingests city-route reference CSV files into a Bronze Delta table `city_routes_bronze`.
    Adds metadata columns (`load_time`, `source_file`) for auditability.

    Parameters:
      once (bool): If True, runs as a batch (trigger once); if False, runs as a continuous stream.
      processing_time (str): Interval for micro-batch triggers when `once=False`.
    """

    # -------------------------------
    # Define expected CSV schema
    # -------------------------------
    schema = "Company_Name STRING, Business_Code STRING, Sector_Code INT, ATOC_Code STRING"

    # -------------------------------
    # Auto Loader stream
    # -------------------------------
    df_stream = (
        spark.readStream
            .format("cloudfiles")                     # Databricks Auto Loader
            .schema(schema)                           # Explicit schema
            .option("cloudFiles.maxFilesPerTrigger", 1)  # 1 file per micro-batch
            .option("cloudFiles.format", "csv")       # Source format
            .option("header", True)                   # First row contains column names
            .load(base_dir_data + "/stations")  # Source path
            .withColumn("load_time", F.current_timestamp())   # Track ingestion time
            .withColumn("source_file", F.input_file_name())  # Track source file
    )

    # -------------------------------
    # Write stream to Bronze Delta table
    # -------------------------------
    

    stream_writer = (
        df_stream.writeStream
            .format("delta")
            .option("checkpointLocation", checkpoint_dir)
            .outputMode("append")
            .queryName("bronze_stations")  # Unique name for monitoring
    )

    # -------------------------------
    # Trigger execution
    # -------------------------------
    if once:
        return stream_writer.trigger(availableNow=True).toTable(f"{catalog}.{schema_1}.station_raw")
    else:
        return stream_writer.trigger(processingTime=processing_time).toTable(f"{catalog}.{schema_1}.station_raw")

In [0]:


# --------------------------------------------------------------------------------------------------
# BRONZE LAYER STREAMING INGESTION FUNCTIONS
# Each function consumes raw files from cloud storage and writes them as Delta tables in append-only mode.
# --------------------------------------------------------------------------------------------------


def consume_station_codes(catalog: str, schema_1: str, checkpoint_dir: str, once=True, processing_time="5 seconds"):
    """
    Ingests city-route reference CSV files into a Bronze Delta table `city_routes_bronze`.
    Adds metadata columns (`load_time`, `source_file`) for auditability.

    Parameters:
      once (bool): If True, runs as a batch (trigger once); if False, runs as a continuous stream.
      processing_time (str): Interval for micro-batch triggers when `once=False`.
    """

    # -------------------------------
    # Define expected CSV schema
    # -------------------------------
    schema = "Company_Name STRING, Business_Code STRING, Sector_Code INT, ATOC_Code STRING"

    # -------------------------------
    # Auto Loader stream
    # -------------------------------
    df_stream = (
        spark.readStream
            .format("cloudfiles")                     # Databricks Auto Loader
            .schema(schema)                           # Explicit schema
            .option("cloudFiles.maxFilesPerTrigger", 1)  # 1 file per micro-batch
            .option("cloudFiles.format", "csv")       # Source format
            .option("header", True)                   # First row contains column names
            .load(base_dir_data + "/stations_code")  # Source path
            .withColumn("load_time", F.current_timestamp())   # Track ingestion time
            .withColumn("source_file", F.input_file_name())  # Track source file
    )

    # -------------------------------
    # Write stream to Bronze Delta table
    # -------------------------------
    

    stream_writer = (
        df_stream.writeStream
            .format("delta")
            .option("checkpointLocation", checkpoint_dir)
            .outputMode("append")
            .queryName("bronze_stations_code")  # Unique name for monitoring
    )

    # -------------------------------
    # Trigger execution
    # -------------------------------
    if once:
        return stream_writer.trigger(availableNow=True).toTable(f"{catalog}.{schema_1}.stations_code_raw")
    else:
        return stream_writer.trigger(processingTime=processing_time).toTable(f"{catalog}.{schema_1}.stations_code_raw")

In [0]:
stream_to_bronze_train(
    topic = "train-movements",
    API_KEY = api_key,
    API_SECRET = api_secret,
    catalog = catalog,
    schema_1 = schema_1,
    checkpoint_dir=TRAIN_CHECKPOINT,
    trigger_once=True
)


In [0]:
stream_to_bronze_weather(
    topic = "weather-updates",
    API_KEY = api_key,
    API_SECRET = api_secret,
    catalog = catalog,
    schema_1 = schema_1,
    checkpoint_dir=WEATHER_CHECKPOINT,
    trigger_once=True
)


In [0]:
consume_city_routes(catalog = catalog, schema_1 = schema_1, checkpoint_dir= CITY_ROUTE_CHECKPOINT, once=True)


In [0]:
consume_station(catalog = catalog, schema_1 = schema_1, checkpoint_dir= STATION_CHECKPOINT, once=True)

In [0]:
consume_station_codes(catalog = catalog, schema_1 = schema_1, checkpoint_dir= STATIONS_CODE_CHECKPOINT, once=True)

In [0]:
# --------------------------------------------------------------------------------------------------
# VALIDATION FUNCTIONS
# Verify that expected record counts exist in Bronze tables after ingestion.
# --------------------------------------------------------------------------------------------------
def assert_zero(catalog, schema_1, table_name, condition):
    """
    Assert that no records match a bad condition.
    """
    print(f"Checking {table_name} for condition: {condition}")

    count = (
        spark.read.table(f"{catalog}.{schema_1}.{table_name}")
        .where(condition)
        .count()
    )

    assert count == 0, f"{count} invalid records found in {table_name} where {condition}"

    print("✔ Passed")


def assert_not_empty(catalog, schema_1, table_name):
    """
    Ensure table is not empty.
    """
    print(f"Checking {table_name} is not empty")

    count = spark.read.table(f"{catalog}.{schema_1}.{table_name}").count()

    assert count > 0, f"{table_name} is empty"

    print(f"✔ {count} records found")

In [0]:


print("\n🔎 Validating Bronze Layer")


assert_not_empty(catalog, schema_1, "train_movements_raw")
assert_not_empty(catalog, schema_1, "weather_raw")
assert_not_empty(catalog, schema_1, "city_routes_raw")
assert_not_empty(catalog, schema_1, "stations_code_raw")
assert_not_empty(catalog, schema_1, "station_raw")

# Only two topics
# assert_zero(catalog, schema_1, "weather_raw", "topic = weather-updates")
# assert_zero(catalog, schema_1, "train_movements_raw", "topic = train-movements")

print("✔ Bronze validation completed")

In [0]:


output_data = {                      # now a STRING, safe for JSON
    "catalog": "dev",
    "schema_1": "bronze",
    "schema_2": "silver",
    "schema_3": "gold"
}

# Store the dictionary (now fully JSON-serializable)
dbutils.jobs.taskValues.set(
    key="bronze_output",
    value=output_data
)